# bitladder: KWS, the full pass of what is still uncomputed

One straight run: Runtime → Run all (T4). Every block is a chain of `run_once`: results and small checkpoints are
pushed to the private results repo after each run, finished runs (any earlier session) are skipped, so after a
disconnect you re-run the notebook and it continues. Blocks, in priority order (times on a T4):

| block | what | why | time |
|---|---|---|---|
| A | pair {on, off} on the big base 128×6, 3 seeds, 3 in-budget branches | second pair for README (3 seeds × 2 pairs) | ~55 min |
| B | thresholds calibrated on Common Voice (`--neg cv`) for both pairs on the big base, 3 seeds | the paper's protocol; skipped if the cache has no `cv_test.npy` | ~40 min |
| C | branch grid on the standard base: epochs × lr × taps | 3d-2, never ran | ~20 min |
| D | methods on 64×4 for seeds 1–2: 4-bit base, TWN branch, KD base + branch | 3f had one seed | ~60 min |
| E | bit ladder on the 12-class model (official test) + layer analysis | CLAUDE.md step 2 | ~35 min |
| F | summary table, push | | 1 min |

**Status after the full pass of 2026-11-03**: blocks A, C, D, E, F are done and in the results repo
(`colab-2026-11-03/`, written up in `docs/results`). **Missing: block B** (thresholds calibrated on Common
Voice), skipped because the cache tarball had no `cv_test.npy`. To get it: upload `kws_cache_cv.tar`
(built locally, 80 MB) to Google Drive, share by link, put its file id into `CV_FILE_ID` in cell 2a, then
Runtime → Run all: every finished run is skipped by `run_once`, only block B trains (18 branches, ~40 min),
block F rebuilds the summary with the `negcv` rows.

**Colab**: Runtime → T4 GPU; Secrets (key icon) → `RESULTS_TOKEN` with notebook access.
**Kaggle**: File → Import notebook → GitHub URL of this file; Settings → Accelerator GPU, Internet ON;
Add-ons → Secrets → `RESULTS_TOKEN`.

In [ ]:
# 1. Code (Colab or Kaggle). Experiments go through run_once(): a crash STOPS the cell, finished runs are skipped on a re-run.
import os, sys
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
REPO = 'https://github.com/Vzzzz/bitladder.git'
if not os.path.exists(f'{WORK}/bitladder'):
    !git clone -q {REPO} {WORK}/bitladder
%cd {WORK}/bitladder
!git pull -q --ff-only && git log -1 --oneline
!pip install -q -e . 2>&1 | grep -iE 'error|failed' || true
for m in [m for m in sys.modules if m == 'bitladder' or m.startswith('bitladder.')]:
    del sys.modules[m]   # a kernel that imported the old code before `git pull` would keep using it
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
from bitladder.notebook import run, run_once, done   # raises ImportError if the install above failed

In [ ]:
# 2a. Prebuilt feature cache from a link-shared file (Google Drive file id or any direct URL).
# Build it locally once: tar -cf kws_cache.tar -C data kws_cache
CACHE_FILE_ID = '1Ho0bzfCwmfxS65FYsntta4HbjkAcQSkP'   # kws_cache.tar on Google Drive (link-shared); or set CACHE_URL
CACHE_URL = ''
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/kws_cache/index.json'):
    if CACHE_FILE_ID:
        !pip install -q gdown && gdown -q {CACHE_FILE_ID} -O {WORK}/kws_cache.tar
    elif CACHE_URL:
        !wget -q -O {WORK}/kws_cache.tar "{CACHE_URL}"
    !tar -xf {WORK}/kws_cache.tar -C data && rm -f {WORK}/kws_cache.tar
# Optional second tarball with the Common Voice test windows (cv_test.npy + index, 80 MB) for `--neg cv` thresholds.
# Build it locally: tar -cf kws_cache_cv.tar -C data kws_cache/cv_test.npy kws_cache/cv_test_index.json; share by link.
CV_FILE_ID = ''   # Google Drive file id of kws_cache_cv.tar; empty = skip (runs that need it are skipped too)
if CV_FILE_ID and not os.path.exists('data/kws_cache/cv_test.npy'):
    !pip install -q gdown && gdown -q {CV_FILE_ID} -O {WORK}/kws_cache_cv.tar
    !tar -xf {WORK}/kws_cache_cv.tar -C data && rm -f {WORK}/kws_cache_cv.tar
!ls -la data/kws_cache
assert os.path.exists('data/kws_cache/index.json'), 'cache missing: gdown failed (quota / sharing); use 2b to rebuild'
print('Common Voice test windows:', os.path.exists('data/kws_cache/cv_test.npy'))

In [ ]:
# S. Results sync: after every run, JSON/MD/PNG/CSV and small checkpoints (<= 2 MB) go to the private results repo.
# Restore brings back EVERY label (newest first), so bases trained in earlier sessions are not retrained and their
# checkpoints serve as KD teachers / branch bases. Token: fine-grained PAT in the notebook secrets as RESULTS_TOKEN
# (Colab: key icon, notebook access ON; Kaggle: Add-ons -> Secrets), Repository access -> bitladder-results,
# Contents: read and write. Fails hard if the sync cannot start; ALLOW_NO_SYNC = True only to run without it.
import datetime
from bitladder.results_sync import ResultsSync
RESULTS_REPO = 'Vzzzz/bitladder-results'
LABEL = f"{'kaggle' if ON_KAGGLE else 'colab'}-{datetime.date.today()}"
ALLOW_NO_SYNC = False

def _secret(name):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    from google.colab import userdata
    return userdata.get(name)

try:
    sync = ResultsSync(RESULTS_REPO, _secret('RESULTS_TOKEN'), LABEL, work_dir='/tmp')
    print('results sync on ->', RESULTS_REPO, '/', LABEL)
    labels = sorted((d.name for d in sync.dir.iterdir() if d.is_dir() and not d.name.startswith('.')), reverse=True)
    print('restored from the repo:', sum(sync.restore('results', label=lab) for lab in labels), 'file(s) from', labels)
except Exception as e:
    msg = f'{type(e).__name__}: {str(e)[:300]}'
    if not ALLOW_NO_SYNC:
        raise RuntimeError('results sync could not start: ' + msg) from e
    print('results sync OFF:', msg)
    class _NoSync:
        def push(self, *a, **k):
            return 0
        def restore(self, *a, **k):
            return 0
    sync = _NoSync()

In [ ]:
# 2. Smoke (~2 min): one epoch of a base and one epoch of a branch into /tmp, every code path the blocks below use.
HAVE_CV = os.path.exists('data/kws_cache/cv_test.npy')
print('Common Voice negatives in the cache:', HAVE_CV)
run('python audio/kws/run.py --keywords yes,no,up,down,left,right,on,off --exclude stop,go --unknown-ratio 1.0 '
    '--epochs 1 --out /tmp/kws_smoke --name smoke')
run('python audio/kws/expand.py --base /tmp/kws_smoke/smoke.json --new stop,go --epochs 1 --k 8 --separable --taps 2 '
    '--scheme uniform --wbits 4 --abits 8 --no-first-last --out /tmp/kws_smoke --name smoke_branch'
    + (' --neg cv' if HAVE_CV else ''))

In [ ]:
# A. Pair {on, off} on the big base (DS-CNN 128x6, fp32, 30 epochs), 3 seeds, the three in-budget branches of 3e. ~55 min.
PAIRS = {'stopgo': ('yes,no,up,down,left,right,on,off', 'stop,go'), 'onoff': ('yes,no,up,down,left,right,stop,go', 'on,off')}
BRANCHES = [('k16ds-t4', '--k 16 --separable --taps 4'), ('k8ds-t3', '--k 8 --separable --taps 3'), ('k8ds-t2', '--k 8 --separable --taps 2')]
BR = '--scheme uniform --wbits 4 --abits 8 --no-first-last'
SEEDS = (0, 1, 2)

def big_base(pair, s):
    base_kw, new = PAIRS[pair]
    name = f'base8_{pair}_b128x6_fp32_s{s}'
    run_once(f'python audio/kws/run.py --keywords {base_kw} --exclude {new} --unknown-ratio 1.0 --channels 128 --blocks 6 '
             f'--epochs 30 --seed {s} --name {name}', name, ckpt=True)
    sync.push('results')
    return name

for s in SEEDS:
    base = big_base('onoff', s)
    for arch, a in BRANCHES:
        n = f'exp_onoff_b128x6_{arch}_u4a8_s{s}'
        run_once(f'python audio/kws/expand.py --base results/{base}.json --new on,off {BR} --epochs 20 --seed {s} {a} --name {n}', n)
        sync.push('results')

In [ ]:
# B. [MISSING after 2026-11-03: needs CV_FILE_ID in cell 2a, see the note on top] Thresholds calibrated on Common Voice (the paper's protocol) instead of GSC words: both pairs, big base, 3 seeds,
#    the same three branches. Needs cv_test.npy in the cache (built by audio/kws/cv_negatives.py); skipped otherwise. ~40 min.
if not HAVE_CV:
    print('skip B: no cv_test.npy in data/kws_cache (rebuild the cache tar with the CV windows)')
else:
    for pair in ('stopgo', 'onoff'):
        for s in SEEDS:
            base = big_base(pair, s)   # restored from an earlier session, or trained now
            for arch, a in BRANCHES:
                n = f'exp_{pair}_b128x6_{arch}_u4a8_negcv_s{s}'
                run_once(f'python audio/kws/expand.py --base results/{base}.json --new {PAIRS[pair][1]} {BR} --neg cv '
                         f'--epochs 20 --seed {s} {a} --name {n}', n)
                sync.push('results')

In [ ]:
# C. Branch grid on the standard base (64x4, 4-bit): epochs x lr x taps, k=16 separable 4-bit, pair {stop, go}, seed 0. ~20 min.
B64 = '--keywords yes,no,up,down,left,right,on,off --exclude stop,go --unknown-ratio 1.0 --channels 64 --blocks 4'
Q = '--epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8'

def std_base(s):
    """fp32 base 64x4 and its 4-bit QAT (both restored from 2026-11-01 for seed 0)."""
    fp, q = f'base8_stopgo_b64x4_fp32_s{s}', f'base8_stopgo_b64x4_u4a8_s{s}'
    run_once(f'python audio/kws/run.py {B64} --epochs 30 --seed {s} --name {fp}', fp, ckpt=True)
    sync.push('results')
    run_once(f'python audio/kws/run.py {B64} {Q} --init results/{fp}.pt --seed {s} --name {q}', q, ckpt=True)
    sync.push('results')
    return fp, q

_, q0 = std_base(0)
GRID_BR = '--new stop,go --k 16 --separable --scheme uniform --wbits 4 --abits 8 --no-first-last'
for ep in (20, 40):
    for lr in ('1e-3', '3e-3', '1e-2'):
        for taps in (2, 4):
            n = f'exp_grid_k16ds-u4a8_e{ep}_lr{lr}_t{taps}_s0'
            run_once(f'python audio/kws/expand.py --base results/{q0}.json {GRID_BR} --epochs {ep} --lr {lr} --taps {taps} --name {n}', n)
            sync.push('results')

In [ ]:
# D. Methods on the standard base for seeds 1-2 (seed 0 is in 3f of 2026-11-01): 4-bit base + k16 separable branch
#    (uniform 4-bit and TWN ternary), KD base from the 128x6 teacher (fp32 then 4-bit) + branch. ~60 min.
BR20 = '--new stop,go --epochs 20 --k 16 --separable --no-first-last'
for s in (1, 2):
    fp, q = std_base(s)
    for arch, a in [('k16ds-u4a8_e20', '--scheme uniform --wbits 4 --abits 8'), ('k16ds_twn-a8', '--scheme twn --abits 8')]:
        n = f'exp_b64x4_{arch}_s{s}'
        run_once(f'python audio/kws/expand.py --base results/{q}.json {BR20} --seed {s} {a} --name {n}', n)
        sync.push('results')
    teacher = big_base('stopgo', s)   # restored from 2026-11-01 (3e), retrained only if its .pt is missing
    kd, kdq = f'base8_stopgo_b64x4kd_fp32_s{s}', f'base8_stopgo_b64x4kd_u4a8_s{s}'
    run_once(f'python audio/kws/run.py {B64} --epochs 30 --seed {s} --teacher results/{teacher}.json --name {kd}', kd, ckpt=True)
    sync.push('results')
    run_once(f'python audio/kws/run.py {B64} {Q} --init results/{kd}.pt --seed {s} --teacher results/{teacher}.json --name {kdq}', kdq, ckpt=True)
    sync.push('results')
    n = f'exp_b64x4kd_k16ds-u4a8_s{s}'
    run_once(f'python audio/kws/expand.py --base results/{kdq}.json {BR20} --seed {s} --scheme uniform --wbits 4 --abits 8 --name {n}', n)
    sync.push('results')

In [ ]:
# E. Bit ladder on the 12-class model (comparable with Hello Edge, official test set): fp32, then QAT from it at
#    uniform 8 / 4 / 2 bits, ternary (absmean and TWN), binary, all with 8-bit activations, first/last layer at 8 bits.
#    Plus the layer analysis of the fp32 model (sensitivity heatmap). ~35 min.
run_once('python audio/kws/run.py --epochs 30 --name kws12_fp32_s0', 'kws12_fp32_s0', ckpt=True)
sync.push('results')
for label, a in [('u8a8', '--scheme uniform --wbits 8 --abits 8'), ('u4a8', '--scheme uniform --wbits 4 --abits 8'),
                 ('u2a8', '--scheme uniform --wbits 2 --abits 8'), ('t158a8', '--scheme ternary --abits 8'),
                 ('twna8', '--scheme twn --abits 8'), ('b1a8', '--scheme binary --abits 8')]:
    n = f'kws12_{label}_s0'
    run_once(f'python audio/kws/run.py --epochs 15 --lr 1e-3 {a} --init results/kws12_fp32_s0.pt --name {n}', n)
    sync.push('results')
run_once('python audio/kws/analyze.py --run results/kws12_fp32_s0.json --split validation --batches 10',
         'kws12_fp32_s0_analysis', show=('|', '#', 'wrote'))
sync.push('results')

In [ ]:
# F. Summary tables: bases (accuracy, bytes) and branches (FRR at FAR 1 %), pushed as summary_bases.csv / summary.csv.
import json, glob, pandas as pd
bases = []
for f in sorted(glob.glob('results/base8_*.json') + glob.glob('results/kws12_*.json')):
    r = json.load(open(f))
    if 'total_bytes' not in r:
        continue   # analysis files
    bases.append({'name': r['name'], 'label': r['label'], 'channels': r['channels'], 'blocks': r['blocks'], 'seed': r['seed'],
                  'teacher': bool(r.get('teacher')), 'params': r['total_params'], 'bytes': r['total_bytes'],
                  'mean_wbits': round(r['mean_wbits'], 2), 'MMAC': round(r['macs'] / 1e6, 1),
                  'test_acc': r['test_acc'], 'official_test_acc': r.get('official_test_acc')})
rows = []
for f in sorted(glob.glob('results/exp_*.json')):
    r = json.load(open(f))
    rows.append({'name': r['name'], 'base': os.path.basename(r['base']).replace('.json', ''), 'new': '-'.join(r['new_words']),
                 'k': r['k'], 'separable': r.get('separable', False), 'taps': r.get('taps', 0), 'neg': r.get('neg', 'gsc'),
                 'seed': r['seed'], 'branch': r['config']['scheme'], 'wbits': r['config']['wbits'], 'epochs': r['epochs'],
                 'branch_params': r['branch_params'], 'branch_bytes': r['branch_bytes'],
                 'frr_ungated': r['test_macro_frr_ungated'], **{f'frr_{w}': v for w, v in r['test_frr_ungated'].items()},
                 'frr_soft': r.get('test_macro_frr_soft'), 'frr_hard': r['test_macro_frr'], 'core_fires_on_new': r['core_fires_on_new']})
pd.DataFrame(bases).to_csv('results/summary_bases.csv', index=False)
df = pd.DataFrame(rows)
df.to_csv('results/summary.csv', index=False)
display(pd.DataFrame(bases))
display(df.groupby(['base', 'new', 'k', 'separable', 'taps', 'neg', 'branch', 'epochs'])['frr_ungated'].agg(['mean', 'std', 'count']))
sync.push('results')
!tar -czf {WORK}/results_json.tgz $(ls results/*.json results/*.md results/*.png results/*.csv 2>/dev/null) && ls -la {WORK}/results_json.tgz